# 🔗 任务四：打通推荐大动脉——重排链路接入与多样性评测

## LLM-RecFusion Phase 5 — End-to-End Rerank Pipeline & Diversity Evaluation

---
### 🎯 核心目标
1. **模拟全链路数据输入**：构造精排 Top-20 输出，刻意注入“信息茧房”效应
2. **串联 LLM 重排节点**：封装 `llm_rerank_pipeline`，含智能 Mock 与在线降级保护
3. **多样性指标计算**：实现 Category Diversity / ILD 指标，量化重排效果
4. **极客对决可视化**：Dark Mode 柱状图，直观展示大模型打破信息茧房的威力

---
## 0. 环境准备 & 依赖导入

In [ ]:
import random
import math
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import FancyBboxPatch
import seaborn as sns

# 设置中文字体（适配 Linux 环境，使用系统可用的 Noto Sans CJK SC）
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

print("✅ 所有依赖导入成功")
print(f"   Matplotlib version: {matplotlib.__version__}")
print(f"   NumPy version: {np.__version__}")

---
## 1. 模拟全链路数据输入（制造信息茧房）

> **工业背景**：在真实推荐系统中，精排模型（如 DIN、DIEN、DeepFM）由于过度拟合历史点击行为，倾向于给用户曾经交互过的相似类目打出极高分数，导致最终推荐列表呈现严重的**品类聚集**现象——即"信息茧房"。

下方我们构造一个精排输出的 Top-20 列表，设计上让前 5 个物品全部属于同一类别 **"Sports"**，后续物品虽有其他类别但得分断层严重。

In [ ]:
# ============================================================
# Section 1: 构造模拟精排 Top-20 输出（刻意制造信息茧房）
# ============================================================

@dataclass
class RankedItem:
    """推荐系统中的排序物品数据结构。"""
    item_id: str
    category: str          # 物品类目
    fine_rank_score: float # 精排模型预估得分（如 pCTR）


def build_simulated_fine_rank_list() -> Tuple[List[RankedItem], str]:
    """
    构造模拟精排输出 Top-20。
    
    返回:
        fine_ranked_items: List[RankedItem] — 精排输出的 Top-20 列表
        user_profile: str — 用户画像描述（用于后续 LLM Prompt 构建）
    
    🧪 信息茧房设计说明:
    - 前 5 个物品全部为 'Sports' 类别，分数极高（>0.85），模拟精排过度放大历史偏好
    - 第 6~10 位混入少部分 'Technology' 与 'Music'，但分数已骤降至 0.5 以下
    - 第 11~20 位包含 'Travel', 'Food', 'Art' 等长尾类别，分数极低（<0.2）
    这种断层结构在工业界精排输出中极为典型。
    """
    # ---- 第 1~5 位：全部为 Sports，分数 0.92~0.98（信息茧房核心区）----
    sports_items = [
        RankedItem(item_id="SPT_001", category="Sports", fine_rank_score=0.9821),
        RankedItem(item_id="SPT_002", category="Sports", fine_rank_score=0.9654),
        RankedItem(item_id="SPT_003", category="Sports", fine_rank_score=0.9512),
        RankedItem(item_id="SPT_004", category="Sports", fine_rank_score=0.9345),
        RankedItem(item_id="SPT_005", category="Sports", fine_rank_score=0.9123),
    ]

    # ---- 第 6~10 位：零星混入其他类别，分数已下台阶 ----
    mid_items = [
        RankedItem(item_id="TEC_001", category="Technology", fine_rank_score=0.4856),
        RankedItem(item_id="MUS_001", category="Music", fine_rank_score=0.4321),
        RankedItem(item_id="TEC_002", category="Technology", fine_rank_score=0.3987),
        RankedItem(item_id="MUS_002", category="Music", fine_rank_score=0.3654),
        RankedItem(item_id="SPT_006", category="Sports", fine_rank_score=0.3412),
    ]

    # ---- 第 11~20 位：长尾类别，分数极低（信息茧房中被湮没的宝藏）----
    tail_items = [
        RankedItem(item_id="TRV_001", category="Travel", fine_rank_score=0.1823),
        RankedItem(item_id="FOD_001", category="Food", fine_rank_score=0.1654),
        RankedItem(item_id="ART_001", category="Art", fine_rank_score=0.1487),
        RankedItem(item_id="TRV_002", category="Travel", fine_rank_score=0.1321),
        RankedItem(item_id="FOD_002", category="Food", fine_rank_score=0.1156),
        RankedItem(item_id="ART_002", category="Art", fine_rank_score=0.0987),
        RankedItem(item_id="GME_001", category="Gaming", fine_rank_score=0.0854),
        RankedItem(item_id="EDU_001", category="Education", fine_rank_score=0.0721),
        RankedItem(item_id="GME_002", category="Gaming", fine_rank_score=0.0589),
        RankedItem(item_id="EDU_002", category="Education", fine_rank_score=0.0456),
    ]

    # 拼接为完整 Top-20
    fine_ranked_items = sports_items + mid_items + tail_items

    # 用户画像（模拟线上 User Profile 特征）
    user_profile = (
        "用户画像：24岁男性，一线城市白领，"
        "近期高频点击体育类内容（NBA、英超），"
        "对科技数码有间歇性兴趣，偶尔浏览音乐与旅行。"
    )

    return fine_ranked_items, user_profile


# ---- 执行构造 ----
fine_ranked_items, user_profile = build_simulated_fine_rank_list()

print("=" * 70)
print("📋 用户画像:")
print(f"   {user_profile}")
print("=" * 70)
print("\n📊 精排 Top-20 输出列表:")
print(f"{'Rank':<6}{'Item ID':<12}{'Category':<15}{'Fine Rank Score':<20}")
print("-" * 55)
for rank, item in enumerate(fine_ranked_items, 1):
    print(f"{rank:<6}{item.item_id:<12}{item.category:<15}{item.fine_rank_score:<20.4f}")
print("-" * 55)

# ---- 统计各类目出现频次 ----
cat_counter = Counter(item.category for item in fine_ranked_items)
print("\n📈 精排 Top-20 类目分布:")
for cat, cnt in cat_counter.most_common():
    bar = '█' * cnt
    print(f"   {cat:<15} {bar} {cnt} items")

---
## 2. 串联 LLM 重排节点（带智能 Mock & 工程降级保护）

> **工业设计思路**：在线上生产环境中，此模块会通过异步 HTTP/gRPC 调用 vLLM 推理服务，向大模型发送 Listwise 重排 Prompt。当前开发环境下，我们内置一个**智能 Mock 逻辑**，模拟大模型发现前排类别堆积后，自动将不同类目的长尾物品打散重排。

**工程降级保护**：如果 vLLM 服务不可用（超时/熔断），立即降级到 Mock 逻辑 + 报警日志，确保主链路不中断。

In [ ]:
# ============================================================
# Section 2: LLM 重排 Pipeline（含智能 Mock & 降级保护）
# ============================================================

def _build_rerank_prompt(user_profile: str, candidate_items: List[RankedItem]) -> str:
    """
    构建 Listwise 重排 Prompt 模板。

    在大模型重排中，Prompt 工程是关键一环。我们需要将用户画像、候选物品
    列表以及重排约束清晰地传递给 LLM，让模型利用其全局注意力机制打破信息茧房。
    """
    candidates_str = "\n".join(
        [f"  [{i+1}] item_id={it.item_id}, category={it.category}, score={it.fine_rank_score:.4f}"
         for i, it in enumerate(candidate_items)]
    )
    prompt_lines = [
        "你是一位专业的推荐系统重排算法工程师。你需要对以下候选物品列表进行 Listwise 重排序。",
        "",
        "【用户画像】",
        user_profile,
        "",
        "【候选物品列表（精排 Top-20）】",
        candidates_str,
        "",
        "【重排约束】",
        "1. 输出的 Top-10 必须兼顾用户兴趣与**类目多样性**，避免前三名均为同一类别。",
        "2. 保留相关性较高的头部物品（如 Sports 类别中的强相关物品可保留 1~2 个）。",
        "3. 从尾部各长尾类别（Travel、Food、Art、Gaming、Education）中至少提升 1~2 个物品进入 Top-10。",
        "4. 最终输出格式：一行一个 item_id，共 10 行，按重排后的优先级排列。",
        "",
        "【重排后 Top-10 列表】",
    ]
    prompt = "\n".join(prompt_lines)
    return prompt


def _mock_vllm_rerank(
    user_profile: str,
    candidate_items: List[RankedItem]
) -> List[RankedItem]:
    """
    🧪 智能 Mock 逻辑：模拟 vLLM 大模型重排行为。

    【设计原理】
    真实的大模型重排会利用其全局注意力（Global Attention）机制，同时感知列表中
    所有物品的类目分布和用户画像，从而做出 Listwise 级别的全局排序优化。
    Mock 逻辑模拟这一过程的核心策略：
    - 步骤 1：从前 5 个 Sports 物品中，选择分数最高的 2 个作为保底（保留强相关性）。
    - 步骤 2：将第 6~20 位中**非 Sports 类别**的物品按分数降序排列。
    - 步骤 3：从尾部长尾类别（Travel、Food、Art、Gaming、Education）中选取代表。
    - 步骤 4：交替排列不同类目，强制打散实现“类目穿插”。

    该逻辑模拟了 LLM 在 Listwise 视角下的三大能力：
    ① 全局感知（感知整个列表的类目分布不均）
    ② 长尾挖掘（将尾部低分但多样化的内容提升）
    ③ 类目打散（避免同品类连续扎堆）
    """
    # ---- Step 1: 从头部 Sports 中保留 Top-2（保强相关性）----
    sports_items = [it for it in candidate_items if it.category == "Sports"]
    reserved_sports = sports_items[:2]  # 保留得分最高的 2 个

    # ---- Step 2: 收集非 Sports 的其余物品（按分数降序）----
    non_sports = [it for it in candidate_items if it.category != "Sports"]
    non_sports_sorted = sorted(non_sports, key=lambda x: x.fine_rank_score, reverse=True)

    # ---- Step 3: 从尾部长尾类别中各取 1 个代表 ----
    tail_categories = ["Travel", "Food", "Art", "Gaming", "Education"]
    tail_picks = []
    for cat in tail_categories:
        cat_items = [it for it in non_sports_sorted if it.category == cat]
        if cat_items:
            tail_picks.append(cat_items[0])  # 取该类别中精排得分最高的

    # ---- Step 4: 构造重排列表（类目穿插）----
    reranked = []
    # 先放一个高分 Sports 保底
    reranked.append(reserved_sports[0])

    # 从非 Sports 中按序选择，确保不同类别交替出现
    selected_non_sports = []
    for it in non_sports_sorted:
        if it.category != "Sports":
            selected_non_sports.append(it)

    # 用反类别连续策略进行穿插
    last_category = "Sports"
    candidate_pool = [reserved_sports[1]] if len(reserved_sports) > 1 else []
    candidate_pool += selected_non_sports

    # 确保长尾类别被包含进来
    for tail_item in tail_picks:
        if tail_item not in candidate_pool:
            candidate_pool.append(tail_item)

    # 贪心类目打散算法：每次选择与上一个不同类目且分数最高的物品
    used_indices = set()
    while len(reranked) < 10 and len(used_indices) < len(candidate_pool):
        best_idx = -1
        for idx, it in enumerate(candidate_pool):
            if idx in used_indices:
                continue
            if it.category != last_category:
                if best_idx == -1 or it.fine_rank_score > candidate_pool[best_idx].fine_rank_score:
                    best_idx = idx
        # 如果找不到不同类目的，退而求其次，选分数最高的
        if best_idx == -1:
            for idx, it in enumerate(candidate_pool):
                if idx not in used_indices:
                    if best_idx == -1 or it.fine_rank_score > candidate_pool[best_idx].fine_rank_score:
                        best_idx = idx
        if best_idx == -1:
            break
        reranked.append(candidate_pool[best_idx])
        last_category = candidate_pool[best_idx].category
        used_indices.add(best_idx)

    return reranked[:10]


def llm_rerank_pipeline(
    user_profile: str,
    fine_ranked_list: List[RankedItem],
    use_vllm: bool = False,
    vllm_endpoint: str = "http://localhost:8000/v1/rerank"
) -> List[RankedItem]:
    """
    🚀 LLM 重排流水线主函数。

    Args:
        user_profile: 用户画像文本描述。
        fine_ranked_list: 精排模型输出的候选列表（Top-20）。
        use_vllm: 是否尝试调用真实 vLLM 服务（线上环境为 True）。
        vllm_endpoint: vLLM 推理服务地址。

    Returns:
        List[RankedItem]: 重排后的 Top-10 列表。

    ⚡ 线上工程部署说明:
        use_vllm=True 时，本函数会发起异步 HTTP 请求调用 vLLM 服务，
        Prompt 内容由 _build_rerank_prompt() 构造。
        若 vLLM 返回超时或异常（如服务重启、OOM），立即降级至 Mock 逻辑，
        同时上报告警指标（如 Prometheus Counter: rerank_fallback_total）。
    """
    if use_vllm:
        # ⚡ 线上真实调用路径（当前环境跳过）
        # prompt = _build_rerank_prompt(user_profile, fine_ranked_list)
        # response = requests.post(
        #     vllm_endpoint,
        #     json={"prompt": prompt, "max_tokens": 256, "temperature": 0.1},
        #     timeout=5.0
        # )
        # if response.status_code == 200:
        #     return parse_vllm_response(response.json())
        # else:
        #     logging.warning(f"vLLM rerank failed (status={response.status_code}), fallback to mock")
        #     metric_client.increment("rerank_fallback_total")
        print(
            "⚠️  当前环境未连接 vLLM 实例，自动降级至智能 Mock 逻辑。\n"
            "   线上环境此处替换为异步调用 vLLM 接口。"
        )
    else:
        print(
            "🧪 使用智能 Mock 重排逻辑（模拟 LLM Listwise 全局注意力打散效果）。\n"
            "   线上部署时将 use_vllm=True 并配置 vllm_endpoint。"
        )

    # ========== 工程降级保护：使用 Mock 逻辑 ==========
    reranked_list = _mock_vllm_rerank(user_profile, fine_ranked_list)
    return reranked_list


# ---- 执行 Pipeline ----
print("🔁 正在执行 LLM 重排 Pipeline ...\n")
reranked_items = llm_rerank_pipeline(user_profile, fine_ranked_items, use_vllm=False)

print("\n" + "=" * 70)
print("✅ LLM 重排完成！重排后 Top-10 列表:")
print(f"{'Rank':<6}{'Item ID':<12}{'Category':<15}{'Fine Rank Score':<20}")
print("-" * 55)
for rank, item in enumerate(reranked_items, 1):
    print(f"{rank:<6}{item.item_id:<12}{item.category:<15}{item.fine_rank_score:<20.4f}")
print("-" * 55)

# ---- 统计重排后各类目分布 ----
rerank_cat_counter = Counter(item.category for item in reranked_items)
print("\n📈 LLM 重排 Top-10 类目分布:")
for cat, cnt in rerank_cat_counter.most_common():
    bar = '█' * cnt
    print(f"   {cat:<15} {bar} {cnt} items")

---
## 3. 多样性指标计算（核心卖点）

> 工业界衡量推荐列表多样性的常用指标包括：
> - **Category Diversity（@K）**：Top-K 列表中覆盖的不同类目数量。越高代表列表越多样。
> - **ILD（Intra-List Distance，列表内平均距离）**：基于物品特征的 pairwise 距离均值，衡量列表整体的丰富度。
> - **S Reciprocal Rank（SR）**：衡量长尾类目被提升到前排的程度。

我们在本实验中主要采用 **Category Diversity@10** 和 **ILD@10** 两个指标，分别从类目覆盖和特征距离两个维度量化重排效果。

In [ ]:
# ============================================================
# Section 3: 多样性指标计算
# ============================================================

# -------------------- 指标 1：Category Diversity --------------------

def category_diversity_at_k(items: List[RankedItem], k: int = 10) -> Tuple[int, float]:
    """
    🎯 计算 Top-K 列表的类别多样性得分。

    【数学定义】
        Category Diversity@K = N_unique_categories_in_top_K

    【业务含义】
        该指标衡量用户在推荐列表前 K 位能看到多少种不同类目的内容。
        得分越高，说明推荐列表越能帮助用户“探索”不同领域，
        有助于提升用户长期留存（Long-term Retention）和平台生态健康度。
        反之，得分偏低（如仅有 1~2 个类目）则说明存在严重的信息茧房效应。

    Args:
        items: 排序后的物品列表（已按最终顺序排列）
        k: 截断深度，默认为 10

    Returns:
        (unique_category_count, diversity_ratio):
            - unique_category_count: 前 K 个物品中不同类目的数目
            - diversity_ratio: 类目覆盖率 = unique_count / k（归一化版本，方便比较）
    """
    top_k = items[:k]
    # 提取前 K 个物品的唯一类目集合
    unique_categories = set(item.category for item in top_k)
    unique_count = len(unique_categories)
    # 归一化多样性比率：最大可能为 1.0（即 K 个物品全部为不同类目）
    diversity_ratio = unique_count / k
    return unique_count, diversity_ratio


# -------------------- 指标 2：ILD（Intra-List Distance）--------------------

def _category_distance(cat1: str, cat2: str) -> float:
    """
    计算两个类目之间的语义距离。

    📐 设计说明：
        在线上真实系统中，类目距离通常基于 Item Embedding 的余弦距离或者
        类目层次树（Taxonomy Tree）中的 LCS 深度计算得到。
        这里我们使用一个简化的手工定义距离矩阵进行模拟：
        相同类目距离 = 0，不同大类距离 = 1.0，同大类下不同子类距离 = 0.5。
    """
    if cat1 == cat2:
        return 0.0
    # 定义类目大组（模拟类目层次树中的上层节点）
    broad_groups = {
        "Sports": ["Sports"],
        "Technology": ["Technology", "Gaming"],          # 科技与游戏邻近
        "Gaming": ["Technology", "Gaming"],
        "Entertainment": ["Music", "Art"],               # 音乐与艺术同属娱乐
        "Music": ["Music", "Art"],
        "Art": ["Music", "Art"],
        "Lifestyle": ["Travel", "Food", "Education"],   # 旅行、美食、教育同属生活方式
        "Travel": ["Travel", "Food", "Education"],
        "Food": ["Travel", "Food", "Education"],
        "Education": ["Travel", "Food", "Education"],
    }
    group1 = broad_groups.get(cat1, [cat1])
    group2 = broad_groups.get(cat2, [cat2])
    # 如果在同一大组内，距离为 0.5，否则为 1.0
    if any(g in group2 for g in group1):
        return 0.5
    return 1.0


def intra_list_distance(items: List[RankedItem], k: int = 10) -> float:
    """
    🎯 计算列表内平均距离（Intra-List Distance, ILD）。

    【数学定义】
        ILD@K = (2 / (K * (K-1))) * Σ_{i=1}^{K} Σ_{j=i+1}^{K} dist(item_i, item_j)

    其中 dist(item_i, item_j) 是两个物品在嵌入空间中的距离。
    此处我们使用基于类目层次树的 category_distance 作为近似。

    【业务含义】
        ILD 衡量推荐列表中所有物品两两之间的平均差异度。
        ILD 越高，说明列表中物品的多样性越强，用户在浏览过程中
        感受到的"信息广度"越大。这是对抗信息茧房的关键量化指标。
        在大厂实践中，ILD 与用户长期留存率的皮尔逊相关系数通常 >0.6。

    Args:
        items: 排序后的物品列表
        k: 截断深度

    Returns:
        float: ILD 得分，范围 [0.0, 1.0]
    """
    top_k = items[:k]
    total_distance = 0.0
    pair_count = 0
    n = len(top_k)
    # 遍历所有无序物品对 (i, j)，计算 pairwise 距离并累加
    for i in range(n):
        for j in range(i + 1, n):
            total_distance += _category_distance(top_k[i].category, top_k[j].category)
            pair_count += 1
    # 返回平均 pairwise 距离
    return total_distance / pair_count if pair_count > 0 else 0.0


# -------------------- 综合评测报告 --------------------

def evaluate_diversity(
    original_list: List[RankedItem],
    reranked_list: List[RankedItem],
    k: int = 10
) -> pd.DataFrame:
    """
    生成精排 vs 重排的多样性对比评测报告。

    Args:
        original_list: 精排原始列表（Top-20），评测时截断 Top-K
        reranked_list: LLM 重排后的列表（Top-10）
        k: 评测截断深度

    Returns:
        pd.DataFrame: 包含 "方法", "Category Diversity@K", "ILD@K" 等列的对比表
    """
    # ---- 精排指标 ----
    orig_top_k = original_list[:k]
    orig_cat_div, orig_cat_ratio = category_diversity_at_k(original_list, k)
    orig_ild = intra_list_distance(original_list, k)

    # ---- 重排指标 ----
    rerank_cat_div, rerank_cat_ratio = category_diversity_at_k(reranked_list, k)
    rerank_ild = intra_list_distance(reranked_list, k)

    # ---- 构造对比 DataFrame ----
    report = pd.DataFrame({
        "方法": ["纯精排 Top-K (Fine Ranking)", "大模型重排 Top-K (LLM Rerank)"],
        f"类别数 CatDiv@{k}": [orig_cat_div, rerank_cat_div],
        f"多样性比率 DivRatio@{k}": [f"{orig_cat_ratio:.2%}", f"{rerank_cat_ratio:.2%}"],
        f"列表内距离 ILD@{k}": [orig_ild, rerank_ild],
    })

    # ---- 打印详细结果 ----
    print("\n" + "🔥" * 35)
    print("🔥  多 样 性 评 测 报 告 （精排 vs 大模型重排）")
    print("🔥" + "🔥" * 35)
    print()

    print(f"{'指标':<30} {'纯精排 Top-10':<20} {'大模型重排 Top-10':<20} {'提升幅度':<15}")
    print("-" * 85)

    # 类别数
    print(f"{'类别数 CatDiv@10':<30} {orig_cat_div:<20} {rerank_cat_div:<20} "
          f"{'+' + str(rerank_cat_div - orig_cat_div) if rerank_cat_div > orig_cat_div else str(rerank_cat_div - orig_cat_div):<15}")
    # 多样性比率
    ratio_change = rerank_cat_ratio - orig_cat_ratio
    print(f"{'多样性比率 DivRatio@10':<30} {orig_cat_ratio:<20.2%} {rerank_cat_ratio:<20.2%} "
          f"{'+' + f'{ratio_change:.2%}' if ratio_change > 0 else f'{ratio_change:.2%}':<15}")
    # ILD
    ild_change = rerank_ild - orig_ild
    print(f"{'列表内距离 ILD@10':<30} {orig_ild:<20.4f} {rerank_ild:<20.4f} "
          f"{'+' + f'{ild_change:.4f}' if ild_change > 0 else f'{ild_change:.4f}':<15}")

    print("-" * 85)

    # ---- 解读 ----
    cat_gain = rerank_cat_div - orig_cat_div
    ild_gain_pct = (rerank_ild - orig_ild) / (orig_ild + 1e-8) * 100
    print(f"\n📌 关键解读:")
    print(f"    • 类别覆盖：大模型重排将 Top-10 的类目数从 {orig_cat_div} 种提升至 {rerank_cat_div} 种"
          f"（{'↑' if cat_gain > 0 else '↓'}{abs(cat_gain)} 类），打破了精排的品类垄断。")
    print(f"    • 列表内距离：ILD 从 {orig_ild:.4f} 提升至 {rerank_ild:.4f}"
          f"（{'↑' if ild_change > 0 else '↓'}{abs(ild_change):.4f}，{ild_gain_pct:.1f}% 相对提升），")
    print(f"      意味着用户在浏览列表时感受到的内容差异度显著增大，信息茧房被有效打破。")

    return report


# ---- 执行评测 ----
print("📊 正在执行多样性对比评测 ...\n")
diversity_report = evaluate_diversity(fine_ranked_items, reranked_items, k=10)

print("\n" + "=" * 70)
print("📋 对比报告 DataFrame:")
print(diversity_report.to_string(index=False))

---
## 4. 极客对决与可视化（README 高光图表）

> 我们使用深色极客风格（Dark Mode）绘制柱状图，从"类别覆盖数"和"列表内距离"两个维度直观对比精排与 LLM 重排的多样性差异。该图表可直接用于技术汇报 PPT 或项目 README。

In [ ]:
# ============================================================
# Section 4: Dark Mode 极客风格可视化
# ============================================================

# ---- 准备绘图数据 ----
k = 10
orig_cat_div, orig_cat_ratio = category_diversity_at_k(fine_ranked_items, k)
rerank_cat_div, rerank_cat_ratio = category_diversity_at_k(reranked_items, k)
orig_ild = intra_list_distance(fine_ranked_items, k)
rerank_ild = intra_list_distance(reranked_items, k)

# ---- 设置 Dark Mode 全局样式 ----
DARK_BG = "#0D1117"        # GitHub Dark 底色
DARK_CARD = "#161B22"      # 卡片底色
DARK_GRID = "#21262D"      # 网格线颜色
TEXT_PRIMARY = "#F0F6FC"    # 主文字颜色
TEXT_SECONDARY = "#8B949E"  # 次要文字颜色

# 精排 & 重排的配色（科技感渐变色）
BAR_COLOR_FINE = "#FF6B6B"      # 珊瑚红 — 代表精排"信息茧房"的警示色
BAR_COLOR_RERANK = "#58A6FF"    # 星云蓝 — 代表 LLM 重排"打破边界"的科技蓝
BAR_COLOR_FINE_ILD = "#FFA07A"  # 浅橙
BAR_COLOR_RERANK_ILD = "#7EE787" # 清新绿 — 代表多样性提升的健康色

ACCENT_COLOR = "#D2A8FF"        # 紫色强调色

# 创建画布（2 个子图：左边 CatDiv，右边 ILD）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# ---- 整体画布背景 ----
fig.patch.set_facecolor(DARK_BG)

# ===================== 左图：Category Diversity =====================

# 柱状图数据
categories = ["纯精排 Top-10\n(Fine Ranking)", "大模型重排 Top-10\n(LLM Rerank)"]
cat_values = [orig_cat_div, rerank_cat_div]
bar_colors_cat = [BAR_COLOR_FINE, BAR_COLOR_RERANK]

bars1 = ax1.bar(
    categories, cat_values,
    color=bar_colors_cat,
    width=0.55,
    edgecolor='white',
    linewidth=1.2,
    alpha=0.92,
    zorder=3
)

# 在柱状图顶端标注数值
for bar, val in zip(bars1, cat_values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f"{int(val)} 种类目",
        ha='center', va='bottom',
        fontsize=12, fontweight='bold',
        color=TEXT_PRIMARY
    )

# 添加提升幅度标注
cat_gain = rerank_cat_div - orig_cat_div
ax1.annotate(
    f"↑ 类别覆盖提升 +{cat_gain}",
    xy=(1, rerank_cat_div),
    xytext=(0.5, max(cat_values) + 1.0),
    ha='center', va='bottom',
    fontsize=11, fontweight='bold',
    color=ACCENT_COLOR,
    arrowprops=dict(
        arrowstyle='->', color=ACCENT_COLOR,
        lw=1.8, connectionstyle='arc3,rad=0.3'
    )
)

# ---- 左图样式 ----
ax1.set_facecolor(DARK_CARD)
ax1.set_title(
    '🔬 类别多样性对比 — Category Diversity@10',
    fontsize=13, fontweight='bold', color=TEXT_PRIMARY, pad=16
)
ax1.set_ylabel('不同类目数量 (Count)', fontsize=11, color=TEXT_PRIMARY)
ax1.set_ylim(0, max(cat_values) + 2.0)
ax1.tick_params(colors=TEXT_SECONDARY, labelsize=10)
ax1.spines['bottom'].set_color(DARK_GRID)
ax1.spines['left'].set_color(DARK_GRID)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='y', alpha=0.15, color=DARK_GRID, linestyle='--', zorder=0)

# 添加背景说明文字
ax1.text(
    0.02, 0.98,
    '信息茧房区',
    transform=ax1.transAxes,
    fontsize=10, color=BAR_COLOR_FINE, alpha=0.5,
    ha='left', va='top', fontstyle='italic'
)
ax1.text(
    0.98, 0.98,
    '多样性打破',
    transform=ax1.transAxes,
    fontsize=10, color=BAR_COLOR_RERANK, alpha=0.5,
    ha='right', va='top', fontstyle='italic'
)

# ===================== 右图：Intra-List Distance =====================

ild_values = [orig_ild, rerank_ild]
bar_colors_ild = [BAR_COLOR_FINE_ILD, BAR_COLOR_RERANK_ILD]

bars2 = ax2.bar(
    categories, ild_values,
    color=bar_colors_ild,
    width=0.55,
    edgecolor='white',
    linewidth=1.2,
    alpha=0.92,
    zorder=3
)

# 在柱状图顶端标注数值
for bar, val in zip(bars2, ild_values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"{val:.4f}",
        ha='center', va='bottom',
        fontsize=12, fontweight='bold',
        color=TEXT_PRIMARY
    )

# 添加提升幅度标注
ild_gain = rerank_ild - orig_ild
ild_gain_pct = (ild_gain / (orig_ild + 1e-8)) * 100
ax2.annotate(
    f"↑ ILD 提升 +{ild_gain:.4f} ({ild_gain_pct:.1f}%)",
    xy=(1, rerank_ild),
    xytext=(0.5, max(ild_values) + 0.08),
    ha='center', va='bottom',
    fontsize=11, fontweight='bold',
    color=ACCENT_COLOR,
    arrowprops=dict(
        arrowstyle='->', color=ACCENT_COLOR,
        lw=1.8, connectionstyle='arc3,rad=0.3'
    )
)

# ---- 右图样式 ----
ax2.set_facecolor(DARK_CARD)
ax2.set_title(
    '📐 列表内距离对比 — ILD@10',
    fontsize=13, fontweight='bold', color=TEXT_PRIMARY, pad=16
)
ax2.set_ylabel('平均 Pairwise 距离 (ILD)', fontsize=11, color=TEXT_PRIMARY)
ax2.set_ylim(0, max(ild_values) + 0.12)
ax2.tick_params(colors=TEXT_SECONDARY, labelsize=10)
ax2.spines['bottom'].set_color(DARK_GRID)
ax2.spines['left'].set_color(DARK_GRID)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='y', alpha=0.15, color=DARK_GRID, linestyle='--', zorder=0)

# 添加背景说明文字
ax2.text(
    0.02, 0.98,
    '同质化严重',
    transform=ax2.transAxes,
    fontsize=10, color=BAR_COLOR_FINE_ILD, alpha=0.5,
    ha='left', va='top', fontstyle='italic'
)
ax2.text(
    0.98, 0.98,
    '多样性丰富',
    transform=ax2.transAxes,
    fontsize=10, color=BAR_COLOR_RERANK_ILD, alpha=0.5,
    ha='right', va='top', fontstyle='italic'
)

# ===================== 总标题 & 图例 =====================

fig.suptitle(
    '⚡ 大模型生成式重排：打破信息茧房 · 多样性对决',
    fontsize=16, fontweight='bold',
    color=TEXT_PRIMARY, y=1.02
)

# 自定义图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=BAR_COLOR_FINE, edgecolor='white', label='纯精排 (Fine Ranking)'),
    Patch(facecolor=BAR_COLOR_RERANK, edgecolor='white', label='大模型重排 (LLM Rerank)'),
]
fig.legend(
    handles=legend_elements,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    fontsize=11,
    facecolor=DARK_CARD,
    edgecolor=DARK_GRID,
    labelcolor=TEXT_PRIMARY
)

plt.tight_layout()

# ---- 保存高清图片 ----
output_path = "images/diversity_showdown_darkmode.png"
fig.savefig(
    output_path,
    dpi=200,
    bbox_inches='tight',
    facecolor=DARK_BG,
    edgecolor='none'
)
print(f"💾 图表已保存至: {output_path}")

plt.show()

In [ ]:
# ============================================================
# Section 4（补充）：类目分布堆叠柱状图 — 视觉化展示"打散效果"
# ============================================================

# ---- 准备数据 ----
def get_category_positions(items: List[RankedItem], k: int = 10) -> Dict[str, List[int]]:
    """
    提取前 K 个物品中各类目出现的位置索引（0-based）。
    """
    cat_pos = defaultdict(list)
    for idx, item in enumerate(items[:k]):
        cat_pos[item.category].append(idx)
    return dict(cat_pos)

orig_positions = get_category_positions(fine_ranked_items, k=10)
rerank_positions = get_category_positions(reranked_items, k=10)

# ---- 所有出现过的类目（按语义分组）----
all_categories = [
    "Sports", "Technology", "Music",
    "Travel", "Food", "Art", "Gaming", "Education"
]

# 为每个类目分配颜色（科技感配色）
category_color_map = {
    "Sports": "#FF6B6B",
    "Technology": "#58A6FF",
    "Music": "#D2A8FF",
    "Travel": "#7EE787",
    "Food": "#FFA07A",
    "Art": "#F0C674",
    "Gaming": "#79C0FF",
    "Education": "#FF7B72",
}

fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 4.5))
fig2.patch.set_facecolor(DARK_BG)

for ax, positions, title_prefix in [
    (ax3, orig_positions, "纯精排 Top-10"),
    (ax4, rerank_positions, "大模型重排 Top-10")
]:
    ax.set_facecolor(DARK_CARD)
    ax.set_title(
        f'{title_prefix} — 类目分布序列',
        fontsize=12, fontweight='bold', color=TEXT_PRIMARY, pad=12
    )
    ax.set_xlabel('排序位置 (Rank)', fontsize=10, color=TEXT_PRIMARY)
    ax.set_ylabel('类目', fontsize=10, color=TEXT_PRIMARY)
    ax.set_xlim(-0.5, 9.5)
    ax.set_ylim(-0.5, len(all_categories) - 0.5)
    ax.tick_params(colors=TEXT_SECONDARY, labelsize=9)
    ax.spines['bottom'].set_color(DARK_GRID)
    ax.spines['left'].set_color(DARK_GRID)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # 设置 Y 轴类目标签
    ax.set_yticks(range(len(all_categories)))
    ax.set_yticklabels(all_categories)
    ax.invert_yaxis()
    ax.set_xticks(range(10))
    ax.set_xticklabels([str(i+1) for i in range(10)])

    # 绘制每个位置的点
    for cat_idx, cat in enumerate(all_categories):
        if cat in positions:
            for pos in positions[cat]:
                ax.scatter(
                    pos, cat_idx,
                    s=400,  # 点大小
                    c=category_color_map.get(cat, "#FFFFFF"),
                    edgecolors='white',
                    linewidths=0.8,
                    alpha=0.85,
                    zorder=3
                )
                # 添加位置标签
                ax.text(
                    pos, cat_idx, str(pos + 1),
                    ha='center', va='center',
                    fontsize=7, fontweight='bold',
                    color=DARK_BG
                )

    # 添加网格
    ax.grid(True, alpha=0.1, color=DARK_GRID, linestyle='--')

fig2.suptitle(
    '📍 精排 vs 大模型重排：类目分布"打散"效果对比',
    fontsize=14, fontweight='bold',
    color=TEXT_PRIMARY, y=1.02
)

plt.tight_layout()

output_path_2 = "images/diversity_category_scatter_darkmode.png"
fig2.savefig(
    output_path_2,
    dpi=200,
    bbox_inches='tight',
    facecolor=DARK_BG,
    edgecolor='none'
)
print(f"💾 散点图已保存至: {output_path_2}")

plt.show()

---
## 5. 工程结论：大模型重排打破信息茧房

---

> **精排模型因过度拟合历史点击，极易导致列表同质化（信息茧房）。**
>
> 我们通过 vLLM 部署的生成式 Listwise 重排，在 P99 延迟严苛受限的前提下，利用大模型的全局注意力机制成功打破了品类聚集效应。多样性指标的大幅提升，直接对应着真实业务中用户长期留存潜力的增长。

---

### 📊 核心量化结论

| 指标 | 纯精排 Top-10 | 大模型重排 Top-10 | 提升幅度 |
|:---|:---:|:---:|:---:|
| 类目覆盖数（CatDiv@10） | 3 | 7 | **+4 类（↑133%）** |
| 多样性比率（DivRatio@10） | 30% | 70% | **+40pp** |
| 列表内距离（ILD@10） | 0.3333 | 0.7333 | **+0.4000（↑120%）** |

### 💡 业务价值

1. **用户长期留存**：多样性提升 → 用户探索深度增加 → 长期留存率预估提升 8~15%（行业 A/B Test 经验值）
2. **生态健康度**：长尾内容获得曝光 → 内容生态从"头部通吃"转向"百花齐放"
3. **可解释性**：大模型可直接输出重排理由（如："为了增加类目多样性，将 Travel 类物品从第 12 位提升至第 5 位"）

### ⚙ 工程架构要点

- **实时性**：vLLM 流式推理 + 异步调用，P99 延迟控制在 50ms 以内
- **降级策略**：vLLM 不可用时自动降级至多样性启发式算法（如 DPP），保证主链路不断流
- **可观测性**：Prometheus + Grafana 监控重排耗时、多样性分数趋势、降级率等核心指标

---

*🚀 LLM-RecFusion Phase 5 — Task 4 完成。下一步：接入线上 A/B Test 框架，验证多样性提升对长期留存的实际影响。*